# Capstone: Run the ICL-as-Bayes Debate

Capstone notebook for the [LLMs & In-Context Learning](https://josephausterweil.github.io/probintro/deep/llms-in-context-learning/) chapter (Part VIII).

**The trial.** "In-context learning is implicit Bayesian inference" is a *contested* research claim. The chapter gave you both sides; here you run them:

- **Part A** — replicate the Xie-style toy: the **exact Bayesian** in-context learner versus a **genuinely learned** predictor (a tiny logistic model you train in this notebook by gradient descent), with the chapter's count-based lookup as a baseline. *(Xie et al. 2022, arXiv:2111.02080; Ye et al. 2024.)*
- **Part B** — implement the **Falck-style martingale test** and run it on *both* predictors. The exact Bayesian must pass; how badly (and how systematically) does the learned model fail? *(Falck et al. 2024, arXiv:2406.00793.)*
- **Part C** *(optional stretch — needs an LLM API that returns token probabilities, or a small open model)* — sketch the same test on a real LLM.

The world, the exact Bayesian, and the pretraining corpus are provided and run end-to-end. The trained model and the martingale test are `TODO`s — each stub states its formula. Work top to bottom.

## 0. Setup (complete — run me)

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

key = jr.PRNGKey(0)
print('JAX ready:', jax.__version__)

## 1. The world and the exact Bayesian (complete — run me)

The chapter's mini-world, where the claim is true *by construction*: documents are conversations in a two-word language (*plip*/*plap* answered with *blue*/*red*), each written under one of two latent **concepts** — plip-is-blue or plip-is-red — with a 10% slip rate, mixed 70/30. The exact Bayesian computes the posterior over the concept and averages the concepts' predictions: the **posterior predictive**.

In [ ]:
# Words: 0 = plip, 1 = plap.   Colors: 0 = blue, 1 = red.
# P_BLUE[c, w] = probability that word w is answered "blue" under concept c.
P_BLUE = jnp.array([[0.9, 0.1],    # concept 0: plip->blue, plap->red (10% slip)
                    [0.1, 0.9]])   # concept 1: the meanings flipped
PRIOR = jnp.array([0.7, 0.3])      # concept-0 documents are more common

WORDS, COLORS = ["plip", "plap"], ["blue", "red"]
prompt = [(0, 0), (1, 1), (0, 0)]  # plip->blue, plap->red, plip->blue

def posterior(examples):
    """P(concept | the prompt's word->color examples)."""
    log_post = jnp.log(PRIOR)
    for w, c in examples:
        p_blue = P_BLUE[:, w]
        log_post += jnp.log(jnp.where(c == 0, p_blue, 1.0 - p_blue))
    post = jnp.exp(log_post - log_post.max())
    return post / post.sum()

def predict_blue(examples, w_query):
    """Posterior predictive: P(next color = blue | query word, examples)."""
    return jnp.sum(posterior(examples) * P_BLUE[:, w_query])

bayes_curve = [float(predict_blue(prompt[:n], 0)) for n in range(4)]
print('exact Bayes, P(blue | plip) after 0..3 examples:',
      [round(p, 4) for p in bayes_curve])

## 2. The pretraining corpus and the count-based baseline (complete — run me)

200,000 documents sampled from the mixture — **no concept labels anywhere in what a predictor sees**. The count-based baseline is the chapter's lookup table: among documents whose opening matches the prompt and whose next word matches the query, the fraction answered blue. Counting-and-normalizing is maximum-likelihood next-token prediction (the cross-entropy minimizer), reached essentially exactly because this world is small enough to memorize.

In [ ]:
N_DOCS, DOC_LEN = 200_000, 4
k1, k2, k3 = jr.split(key, 3)
concept = jr.choice(k1, 2, (N_DOCS,), p=PRIOR)              # latent, per document
words = jr.choice(k2, 2, (N_DOCS, DOC_LEN))
p_blue = P_BLUE[concept[:, None], words]
colors = (jr.uniform(k3, (N_DOCS, DOC_LEN)) > p_blue).astype(jnp.int32)   # 0 = blue

def corpus_predict_blue(examples, w_query):
    """Count-based predictor (the chapter's): match the prompt prefix, count blues."""
    n = len(examples)
    match = words[:, n] == w_query
    for i, (w, c) in enumerate(examples):
        match &= (words[:, i] == w) & (colors[:, i] == c)
    return jnp.sum(match & (colors[:, n] == 0)) / jnp.sum(match)

count_curve = [float(corpus_predict_blue(prompt[:n], 0)) for n in range(4)]

ns = range(4)
plt.figure(figsize=(6, 4))
plt.plot(ns, bayes_curve, 'o-', label='exact Bayes (posterior predictive)')
plt.plot(ns, count_curve, 's--', label='count-based (learned from corpus)')
plt.axhline(0.9, color='gray', lw=1, ls=':', label='0.9 slip ceiling')
plt.xlabel('number of prompt examples'); plt.ylabel('P(blue | plip)')
plt.xticks(list(ns)); plt.legend(); plt.title('In-context sharpening')
plt.tight_layout(); plt.show()

print('count-based curve:', [round(p, 4) for p in count_curve])

## 3. Part A — a genuinely *learned* predictor — TODO

The lookup table memorizes contexts; a real model *learns parameters*. Train the tiniest possible parametric next-token predictor: **logistic regression** on context features.

**Features** (provided): for a context of examples plus a query word, a length-10 vector = the **counts** of the four (word, color) pair types **crossed with the query word** (8 numbers — the count block for the other query word is zeroed), plus a **one-hot** of the query word. Why the crossing? The *same* context pair is evidence in *opposite directions* depending on the question: plip→blue pushes toward "blue" when the query is plip and toward "red" when the query is plap. Plain count features would cancel across the two query words (try it!); the interaction lets each query read the counts with its own signs. Counts are otherwise the right currency — for exchangeable examples the exact posterior log-odds are *linear in the counts* for a fixed query. What a sigmoid-of-linear model still *cannot* represent is the clamp to the 0.1/0.9 slip range — remember that for Part B.

**Your job**: minimize the cross-entropy of the next color over every prefix of every corpus document, by gradient descent with `jax.grad` — the same loss-goes-downhill loop as the [From Rules to Weights](https://josephausterweil.github.io/probintro/deep/from-rules-to-weights/) chapter.

In [ ]:
def context_features(examples, w_query):
    """Length-10 features: the 4 pair-type counts crossed with the query word (8),
    plus a one-hot of the query word (2)."""
    counts = jnp.zeros(4)
    for w, c in examples:
        counts = counts.at[2 * w + c].add(1.0)
    query = jnp.zeros(2).at[w_query].set(1.0)
    inter = jnp.outer(query, counts).reshape(-1)      # counts, per query word
    return jnp.concatenate([inter, query])

def make_training_set(words, colors):
    """Every prefix of every document -> (features, next-color-is-blue) pairs."""
    n_docs, doc_len = words.shape
    pair_ids = 2 * words + colors                                  # values 0..3
    onehot = jax.nn.one_hot(pair_ids, 4)                           # (docs, len, 4)
    csum = jnp.cumsum(onehot, axis=1)
    counts = jnp.concatenate([jnp.zeros((n_docs, 1, 4)), csum[:, :-1]], axis=1)
    query = jax.nn.one_hot(words, 2)                               # (docs, len, 2)
    inter = (query[:, :, :, None] * counts[:, :, None, :]).reshape(n_docs, doc_len, 8)
    feats = jnp.concatenate([inter, query], axis=-1).reshape(-1, 10)
    targets = (colors == 0).astype(jnp.float32).reshape(-1)
    return feats, targets

feats, targets = make_training_set(words, colors)
print('training set:', feats.shape, targets.shape)   # 800,000 prefixes

def train_logistic(feats, targets, steps=300, lr=0.5):
    """Return params {'v': (10,), 'b': ()} minimizing next-color cross-entropy."""
    # TODO:
    #   1. def nll(params):
    #          p = jax.nn.sigmoid(feats @ params['v'] + params['b'])
    #          return -jnp.mean(targets * jnp.log(p + 1e-9)
    #                           + (1 - targets) * jnp.log(1 - p + 1e-9))
    #   2. params = {'v': jnp.zeros(10), 'b': jnp.array(0.0)}
    #   3. loop `steps` times: params = params - lr * jax.grad(nll)(params)
    #      (use jax.tree.map to update both entries at once)
    #   4. return params
    raise NotImplementedError

def logistic_predict_blue(params, examples, w_query):
    """P(blue | query word, examples) under the trained logistic model."""
    f = context_features(examples, w_query)
    return jax.nn.sigmoid(jnp.dot(f, params['v']) + params['b'])

# When train_logistic works, compare all three curves:
# params = train_logistic(feats, targets)
# for n in range(4):
#     print(f"n={n}: Bayes {float(predict_blue(prompt[:n], 0)):.4f}"
#           f"  count {float(corpus_predict_blue(prompt[:n], 0)):.4f}"
#           f"  logistic {float(logistic_predict_blue(params, prompt[:n], 0)):.4f}")

## 4. Part B — the martingale test — TODO

The Falck et al. (2024) signature: for **exchangeable** examples, a Bayesian's running prediction is a **martingale** — today's prediction equals the expected value of tomorrow's, averaged over the model's *own* forecast of the next example. Implement the gap and run it on **all three** predictors.

**Interpretation questions** (answer in a markdown cell):
1. The exact Bayesian's gap should be floating-point zero. Is it?
2. The count-based model's gap is nonzero. Is that *sampling noise* (a finite corpus) or *systematic drift*? How would you tell? (Hint: regenerate the corpus with different seeds — `jr.PRNGKey(1), (2), ...` — and see whether the gap keeps its sign or flips at random.)
3. The logistic model's gap should be larger and *systematic* — a sigmoid-of-linear model cannot represent the exact posterior predictive's 0.1/0.9 clamps (watch its curve overshoot the 0.9 slip ceiling as examples accumulate). Falck et al.'s point in miniature: a trained approximation of a Bayesian is not a Bayesian, and the martingale test can tell.

In [ ]:
def martingale_gap(predict_fn, examples, w_query):
    """|E[tomorrow's prediction] - today's prediction| under predict_fn.

    predict_fn(examples, w_query) -> P(next color = blue). The expectation runs
    over the four possible next exchanges: the next word is uniform (0.5 each),
    and its color is distributed according to predict_fn's OWN forecast.
    """
    # TODO:
    #   today = predict_fn(examples, w_query)
    #   total = 0.0
    #   for w_next in (0, 1):
    #       p_b = predict_fn(examples, w_next)
    #       for c_next, p_c in ((0, p_b), (1, 1.0 - p_b)):
    #           total += 0.5 * p_c * predict_fn(examples + [(w_next, c_next)], w_query)
    #   return abs(float(total - today))
    raise NotImplementedError

# When implemented (and Part A's params exist), run the trial:
# for n in range(3):
#     print(f"n={n}:"
#           f"  Bayes {martingale_gap(predict_blue, prompt[:n], 0):.6f}"
#           f"  count {martingale_gap(corpus_predict_blue, prompt[:n], 0):.6f}"
#           f"  logistic {martingale_gap(lambda e, w: logistic_predict_blue(params, e, w), prompt[:n], 0):.6f}")

## 5. Part C (optional stretch) — the same test on a real LLM

**This part is optional and needs resources beyond this notebook**: either an LLM API that returns token log-probabilities, or a small open model you can run locally (a 0.5–1B-parameter causal LM is plenty). Skip it freely; Parts A–B are the complete lab.

The idea: encode the plip/plap game as literal text, read the model's next-token probabilities for `blue` vs `red`, renormalize over those two, and you have a `predict_fn` you can hand to your `martingale_gap` **unchanged**. Falck et al. did exactly this kind of read-out (on exchangeable synthetic streams) and found systematic violations.

In [ ]:
# Sketch (local open model via the `transformers` library) -- intentionally
# commented out: it downloads a model and needs torch. Uncomment to run.
#
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
# MODEL = "Qwen/Qwen2.5-0.5B"        # any small BASE causal LM
# tok = AutoTokenizer.from_pretrained(MODEL)
# lm = AutoModelForCausalLM.from_pretrained(MODEL)
#
# def llm_predict_blue(examples, w_query):
#     lines = [f"{WORDS[w]} -> {COLORS[c]}" for w, c in examples]
#     text = "\n".join(lines + [f"{WORDS[w_query]} ->"])
#     ids = tok(text, return_tensors="pt").input_ids
#     with torch.no_grad():
#         logits = lm(ids).logits[0, -1]
#     blue = tok(" blue", add_special_tokens=False).input_ids[0]
#     red = tok(" red", add_special_tokens=False).input_ids[0]
#     two = torch.softmax(logits[[blue, red]], dim=0)   # renormalize over {blue, red}
#     return float(two[0])
#
# # Then: chart llm_predict_blue's sharpening curve, and run martingale_gap on it.
#
# Caveats to report with any result:
#   - tokenization: made-up words like "plip" may split into multiple tokens;
#   - renormalizing over {blue, red} ignores probability placed on other tokens;
#   - instruction-tuned (chat) models add RLHF-shaped behavior on top of pure
#     next-token prediction -- prefer a BASE model for the cleanest test.
print("Part C is optional -- see the commented sketch in this cell.")

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*